# Step 2 - Extract Vector Data from PDF as GeoJSON

The source PDF is **100% vector** content:
- 2,846 drawing paths (oil blocks, fields, boundaries, pipelines, land/water)
- 783 text labels (OML/OPL IDs, field names, licensees, cities)
- Projection: **World Equidistant Cylindrical** (x→longitude, y→latitude are linear)

This notebook extracts all paths and labels into three GeoJSON FeatureCollections:
- **areas.geojson** - Filled polygons (oil blocks, oil fields, land, water)
- **lines.geojson** - Stroked paths (boundaries, pipelines, contours)
- **labels.geojson** - Text labels as points (block IDs, field names, cities)

In [ ]:
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from pdf_to_geojson import extract_geojson

In [ ]:
PDF_PATH = os.path.join("..", "data", "Map of Nigeria Oil Field.pdf")
OUTPUT_DIR = os.path.join(".", "output")

results = extract_geojson(PDF_PATH, OUTPUT_DIR)

# Page info
page = results["page"]
print(f"Page size:       {page['page_width_pt']} x {page['page_height_pt']} points")
print(f"Projection:      {page['projection']}")
print(f"Total drawings:  {page['total_drawings']}")
print(f"Total text blocks: {page['total_text_blocks']}")
print()

# Output summary
for name in ("areas", "lines", "labels"):
    r = results[name]
    print(f"{name}.geojson: {r['features']:>5} features  ({r['file_size_mb']:.2f} MB)")

## Classification breakdown

Areas are classified by their fill color according to the map legend.

In [ ]:
import json

with open(os.path.join(OUTPUT_DIR, "areas.geojson"), encoding="utf-8") as f:
    areas = json.load(f)

# Count features by classification
class_counts: dict[str, int] = {}
for feat in areas["features"]:
    cls = feat["properties"].get("class", "No fill")
    class_counts[cls] = class_counts.get(cls, 0) + 1

print("Area polygons by classification:")
print("-" * 42)
for cls, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"  {cls:<30s} {count:>5}")
print(f"  {'TOTAL':<30s} {sum(class_counts.values()):>5}")

In [ ]:
with open(os.path.join(OUTPUT_DIR, "labels.geojson"), encoding="utf-8") as f:
    labels = json.load(f)

# Count labels by font
font_counts: dict[str, int] = {}
for feat in labels["features"]:
    font = feat["properties"]["font"]
    font_counts[font] = font_counts.get(font, 0) + 1

print("Text labels by font:")
print("-" * 42)
for font, count in sorted(font_counts.items(), key=lambda x: -x[1]):
    print(f"  {font:<30s} {count:>5}")
print(f"  {'TOTAL':<30s} {sum(font_counts.values()):>5}")

# Count labels by color
color_counts: dict[str, int] = {}
for feat in labels["features"]:
    color = feat["properties"]["color"]
    color_counts[color] = color_counts.get(color, 0) + 1

print("\nText labels by color:")
print("-" * 42)
for color, count in sorted(color_counts.items(), key=lambda x: -x[1]):
    print(f"  {color:<30s} {count:>5}")

In [ ]:
# Sample: first 10 oil field labels (blue bold text = field names)
print("Sample oil field labels (blue bold text):")
print("-" * 60)
count = 0
for feat in labels["features"]:
    p = feat["properties"]
    if p["color"] == "#0000FF" and p["font"] == "Helvetica-Bold":
        lon, lat = feat["geometry"]["coordinates"]
        print(f"  {p['text']:<25s}  lon={lon:.4f}  lat={lat:.4f}")
        count += 1
        if count >= 10:
            break